# 08 — Extracción de frecuencias POS (Stanza)

Equivalente al notebook `10_extraer_frecuencias_POS.ipynb` de Karen.

Extrae lemas de verbos, adjetivos y sustantivos del subcorpus del COM-B usando Stanza.

**Entrada:** `comb_tweets_final.parquet`, `corpus_cleaned.parquet`  
**Salida:** `verbos_comb_stanza.parquet`, `adjetivos_comb_stanza.parquet`, `sustantivos_comb_stanza.parquet`

In [7]:
# ============================================================
# CELL 0 — CONFIG
# ============================================================
from pathlib import Path

DATA_PROCESSED = Path(r'C:\Users\afpue\Documents\GitHub\icare\kMetodo\resultadosCOMB')

print('[CONFIG] OK')
print(f'  DATA_PROCESSED : {DATA_PROCESSED.resolve()}')


[CONFIG] OK
  DATA_PROCESSED : C:\Users\afpue\Documents\GitHub\icare\kMetodo\resultadosCOMB


In [8]:
# ============================================================
# CELL 1 — IMPORTS Y CARGA
# Equivalente a celdas 0-2 del paper
# ============================================================
import pandas as pd
import stanza
from collections import Counter
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Subcorpus del COM-B (salida de NB04)
# Tiene columnas: chunk_id, id_doc, texto_chunk, COMB_*, score_max, subcat_max,
#                 etiqueta_comb, categoria_detectada
tweets_comp  = pd.read_parquet(DATA_PROCESSED / 'comb_tweets_final.parquet')

# Columnas de interes (equiv. al df_filtrado de Karen)
cols_base = ['chunk_id', 'id_doc', 'texto_chunk',
             'etiqueta_comb', 'categoria_detectada', 'subcat_max']
cols_base = [c for c in cols_base if c in tweets_comp.columns]

df_comp  = tweets_comp[cols_base].copy()

print(f'Tweets del COM-B: {len(df_comp):,}')
print(f'Columnas       : {list(df_comp.columns)}')
df_comp.head(2)


Tweets del COM-B: 12,254
Columnas       : ['chunk_id', 'id_doc', 'texto_chunk', 'etiqueta_comb', 'categoria_detectada', 'subcat_max']


,chunk_id,id_doc,texto_chunk,etiqueta_comb,categoria_detectada,subcat_max
0,5_1,5,Local Gestionan regidores cajas para intubació...,1,COMB_Oportunidad_fisica,COMB_Oportunidad_fisica
1,6_1,6,La otra semana ya saldrá el aumento de contagi...,1,COMB_Oportunidad_fisica,COMB_Oportunidad_fisica


In [9]:
# ============================================================
# CELL 2 — DESCARGAR MODELO STANZA (solo primera vez)
# ============================================================
# stanza.download('es')   # descomentar si es la primera ejecución
print('Stanza listo. Si es la primera vez, descomenta stanza.download("es") arriba.')


Stanza listo. Si es la primera vez, descomenta stanza.download("es") arriba.


In [10]:
# ============================================================
# CELL 3 — INICIALIZAR PIPELINE
# ============================================================
nlp_stanza = stanza.Pipeline(
    lang="es",
    processors="tokenize,pos,lemma",
    use_gpu=True   # cambiar a True si hay GPU disponible
)
print('Pipeline Stanza inicializado.')


2026-07-14 07:43:16 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2026-07-14 07:43:17 INFO: Downloaded file to C:\Users\afpue\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources\resources.json
2026-07-14 07:43:17 WARNING: Language es package default expects mwt, which has been added
2026-07-14 07:43:17 INFO: Loading these models for language: es (Spanish):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |

2026-07-14 07:43:17 INFO: Using device: cuda
2026-07-14 07:43:17 INFO: Loading: tokenize
2026-07-14 07:43:17 INFO: Loading: mwt
2026-07-14 07:43:17 INFO: Loading: pos
2026-07-14 07:43:18 INFO: Loading: lemma
2026-07-14 07:43:19 INFO: Done loading processors!


Pipeline Stanza inicializado.


In [11]:
# ============================================================
# CELL 4 — FUNCIÓN DE EXTRACCIÓN POS
# (Replica exacta de Karen)
# ============================================================
def extraer_pos_frecuencias_stanza(textos):
    """
    Procesa una lista de textos con Stanza.
    Devuelve tres listas de dicts {lema: frecuencia}:
      verbos_frec, adjetivos_frec, sustantivos_frec
    """
    verbos_frec, adjetivos_frec, sustantivos_frec = [], [], []

    for texto in tqdm(textos, desc="POS Stanza"):
        try:
            if not isinstance(texto, str) or not texto.strip():
                verbos_frec.append({})
                adjetivos_frec.append({})
                sustantivos_frec.append({})
                continue

            doc = nlp_stanza(texto)
            verbos, adjetivos, sustantivos = [], [], []

            for sent in doc.sentences:
                for w in sent.words:
                    if not w.lemma or not w.lemma.isalpha():
                        continue
                    upos = w.upos
                    lema = w.lemma.lower()
                    if upos == "VERB":
                        verbos.append(lema)
                    elif upos == "ADJ":
                        adjetivos.append(lema)
                    elif upos == "NOUN":
                        sustantivos.append(lema)

            verbos_frec.append(dict(Counter(verbos)))
            adjetivos_frec.append(dict(Counter(adjetivos)))
            sustantivos_frec.append(dict(Counter(sustantivos)))

        except Exception as e:
            print(f"Error: {e}")
            verbos_frec.append({})
            adjetivos_frec.append({})
            sustantivos_frec.append({})

    return verbos_frec, adjetivos_frec, sustantivos_frec


In [12]:
# ============================================================
# CELL 5 — EJECUTAR EXTRACCIÓN
# (Puede tardar 30-60 min dependiendo del hardware)
# ============================================================

# Karen usa 'texto_chunk' — aqui tambien
textos = df_comp['texto_chunk'].tolist()
verbos_frec, adjetivos_frec, sustantivos_frec = extraer_pos_frecuencias_stanza(textos)

print(f'Extraccion completada para {len(textos):,} tweets.')


POS Stanza: 100%|██████████| 12254/12254 [40:26<00:00,  5.05it/s]

Extraccion completada para 12,254 tweets.


In [13]:
# ============================================================
# CELL 6 — CONSTRUIR DATAFRAMES Y GUARDAR
# Equivalente a celdas 7-9 del paper
# Karen guarda xlsx; aqui guardamos parquet (mejor para dicts)
# ============================================================

# Verbos
df_verbos = df_comp[cols_base].copy()
df_verbos['verbos_lemas_frecuencias'] = verbos_frec
df_verbos.to_parquet(DATA_PROCESSED / 'verbos_comb_stanza.parquet', index=False)
print('[GUARDADO] verbos_comb_stanza.parquet')

# Adjetivos
df_adjetivos = df_comp[cols_base].copy()
df_adjetivos['adjetivos_lemas_frecuencias'] = adjetivos_frec
df_adjetivos.to_parquet(DATA_PROCESSED / 'adjetivos_comb_stanza.parquet', index=False)
print('[GUARDADO] adjetivos_comb_stanza.parquet')

# Sustantivos
df_sustantivos = df_comp[cols_base].copy()
df_sustantivos['sustantivos_lemas_frecuencias'] = sustantivos_frec
df_sustantivos.to_parquet(DATA_PROCESSED / 'sustantivos_comb_stanza.parquet', index=False)
print('[GUARDADO] sustantivos_comb_stanza.parquet')


[GUARDADO] verbos_comb_stanza.parquet
[GUARDADO] adjetivos_comb_stanza.parquet
[GUARDADO] sustantivos_comb_stanza.parquet


In [14]:
# ============================================================
# CELL 7 — VERIFICACIÓN RÁPIDA
# ============================================================
# Top 20 verbos más frecuentes en el subcorpus del COM-B
from collections import Counter

total_verbos = Counter()
for d in verbos_frec:
    total_verbos.update(d)

total_sust = Counter()
for d in sustantivos_frec:
    total_sust.update(d)

total_adj = Counter()
for d in adjetivos_frec:
    total_adj.update(d)

print('Top 20 VERBOS:')
print(total_verbos.most_common(20))
print()
print('Top 20 SUSTANTIVOS:')
print(total_sust.most_common(20))
print()
print('Top 20 ADJETIVOS:')
print(total_adj.most_common(20))

print()
print('Notebook 08 completado.')
print('Siguiente -> 09_descriptivos_subcategorias_comb.ipynb')


Top 20 VERBOS:
[('tener', 3033), ('haber', 2128), ('hacer', 2046), ('ir', 1633), ('decir', 1107), ('dar', 980), ('seguir', 851), ('saber', 774), ('ver', 727), ('salir', 691), ('atender', 655), ('evitar', 610), ('querer', 607), ('pasar', 583), ('llegar', 547), ('contagiar', 539), ('creer', 518), ('morir', 473), ('cuidar', 411), ('tomar', 394)]

Top 20 SUSTANTIVOS:
[('pandemia', 1834), ('vacuna', 1772), ('contagio', 1525), ('virus', 1367), ('salud', 1304), ('hospital', 1151), ('paciente', 1135), ('persona', 1075), ('gente', 982), ('caso', 918), ('día', 859), ('medida', 774), ('país', 707), ('casa', 620), ('prueba', 572), ('enfermedad', 544), ('cama', 521), ('médico', 508), ('covid', 485), ('gobierno', 471)]

Top 20 ADJETIVOS:
[('sanitario', 551), ('médico', 493), ('buen', 412), ('contagiado', 387), ('nuevo', 355), ('mejor', 341), ('público', 332), ('positivo', 304), ('primero', 301), ('mayor', 294), ('necesario', 249), ('importante', 240), ('posible', 218), ('alto', 198), ('social', 181)